# ============================================
# MODULE 1: IMPORTS AND CONFIGURATION
# ============================================
# Description: Import all necessary libraries
# - numpy: numerical operations
# - matplotlib: plotting
# - random: random number generation
# - scipy.stats: statistical functions
# ============================================


In [5]:
import numpy as np
import matplotlib.pyplot as plt
import random
from scipy.stats import norm, entropy
import itertools

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# ============================================
# MODULE 2: PAM MODULATION/DEMODULATION
# ============================================
# Description: Implement PAM-128 (M=2^7=128)
# Input: 7-bit binary words (0-127 decimal)
# Output: Real-valued symbols for AWGN channel
# ============================================

In [6]:
def bits_to_pam(bits):
    """
    Convert 7-bit array to PAM symbol
    Input: bits - list/array of 7 bits (0/1)
    Output: float - PAM symbol value
    """
    # Convert bits to integer (0 to 127)
    decimal_value = 0
    for i, bit in enumerate(bits):
        decimal_value += bit * (2 ** (6 - i))  # MSB first

    # Map to PAM: values from -1 to 1 with equal spacing
    # For M=128, spacing = 2/127 ≈ 0.01575
    M = 128
    pam_value = -1 + (2 * decimal_value) / (M - 1)
    return pam_value

def pam_to_bits(pam_value):
    """
    Convert PAM symbol back to 7-bit array
    Input: pam_value - float (received symbol)
    Output: list of 7 bits (0/1)
    """
    M = 128
    # Clip to valid range
    pam_value = np.clip(pam_value, -1, 1)

    # Convert back to integer
    decimal_value = int(round(((pam_value + 1) * (M - 1)) / 2))
    decimal_value = np.clip(decimal_value, 0, M - 1)

    # Convert to 7-bit binary
    bits = []
    for i in range(6, -1, -1):
        bits.append((decimal_value >> i) & 1)

    return bits

# Test the modulation/demodulation
print("Testing PAM-128 modulation/demodulation:")
test_bits = [1, 0, 0, 1, 0, 0, 0]  # 1001000
pam = bits_to_pam(test_bits)
recovered_bits = pam_to_bits(pam)
print(f"Original bits: {test_bits}")
print(f"PAM symbol: {pam:.4f}")
print(f"Recovered bits: {recovered_bits}")
print(f"Success: {test_bits == recovered_bits}")
print()

Testing PAM-128 modulation/demodulation:
Original bits: [1, 0, 0, 1, 0, 0, 0]
PAM symbol: 0.1339
Recovered bits: [np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0)]
Success: True



# ============================================
# MODULE 3: AWGN CHANNEL IMPLEMENTATION
# ============================================
# Description: Implement AWGN channel for legitimate user (Bob) and eavesdropper (Eve)
# Formula: y = x + noise
# Noise variance: N0 = P * 10^(-SNR_dB/10)
# For PAM, we assume P = 1 (normalized power)
# ============================================

In [7]:
class AWGNChannel:
    """
    AWGN Channel class
    SNR_dB: Signal-to-Noise ratio in decibels
    """
    def __init__(self, SNR_dB):
        self.SNR_dB = SNR_dB
        # Convert dB to linear
        self.SNR_linear = 10 ** (SNR_dB / 10)
        # Noise variance = 1/SNR (assuming signal power = 1)
        self.noise_variance = 1 / self.SNR_linear
        self.noise_std = np.sqrt(self.noise_variance)

    def transmit(self, pam_symbol):
        """
        Transmit PAM symbol through AWGN channel
        Input: pam_symbol - float (transmitted symbol)
        Output: float (received symbol)
        """
        noise = np.random.normal(0, self.noise_std)
        return pam_symbol + noise

# Test AWGN channel
print("Testing AWGN Channel:")
bob_channel = AWGNChannel(SNR_dB=10)  # 10 dB SNR
eve_channel = AWGNChannel(SNR_dB=5)   # 5 dB SNR

test_pam = 0.5
bob_output = bob_channel.transmit(test_pam)
eve_output = eve_channel.transmit(test_pam)
print(f"Input PAM: {test_pam}")
print(f"Bob output (SNR=10dB): {bob_output:.4f}")
print(f"Eve output (SNR=5dB): {eve_output:.4f}")
print()

Testing AWGN Channel:
Input PAM: 0.5
Bob output (SNR=10dB): 0.6571
Eve output (SNR=5dB): 0.4222



# ============================================
# MODULE 4: WIRETAP AWGN CHANNEL (COMPLETE)
# ============================================
# Description: Complete wiretap channel with legitimate and eavesdropper paths
# Input: 7-bit words
# Output: Bob's received bits, Eve's received bits
# ============================================

In [8]:
class WiretapAWGNChannel:
    """
    Complete wiretap AWGN channel
    - Input: 7-bit word
    - Output: (y_bits, z_bits) - Bob and Eve received bits
    """
    def __init__(self, SNR_B_dB, SNR_E_dB):
        self.bob_channel = AWGNChannel(SNR_B_dB)
        self.eve_channel = AWGNChannel(SNR_E_dB)
        self.SNR_B = SNR_B_dB
        self.SNR_E = SNR_E_dB

    def transmit(self, x_bits):
        """
        Transmit 7-bit word through both channels
        """
        # Modulate
        pam_symbol = bits_to_pam(x_bits)

        # Transmit through both channels
        y_pam = self.bob_channel.transmit(pam_symbol)
        z_pam = self.eve_channel.transmit(pam_symbol)

        # Demodulate
        y_bits = pam_to_bits(y_pam)
        z_bits = pam_to_bits(z_pam)

        return y_bits, z_bits

# Test
print("Testing Complete Wiretap AWGN Channel:")
channel = WiretapAWGNChannel(SNR_B_dB=15, SNR_E_dB=10)
x = [1, 0, 0, 1, 0, 0, 0]
y, z = channel.transmit(x)
print(f"Input x: {x}")
print(f"Bob received y: {y}")
print(f"Eve received z: {z}")
print(f"Bob errors: {sum(1 for i in range(7) if x[i] != y[i])}")
print(f"Eve errors: {sum(1 for i in range(7) if x[i] != z[i])}")
print()

Testing Complete Wiretap AWGN Channel:
Input x: [1, 0, 0, 1, 0, 0, 0]
Bob received y: [np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1)]
Eve received z: [np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1)]
Bob errors: 3
Eve errors: 5



# ============================================
# MODULE 5: VERIFICATION OF IMPLEMENTATION
# ============================================
# Description: Verify correctness by transmitting long sequence
# Compare symbol error ratio with sqrt(SNR ratio)
# ============================================

In [9]:
def verify_awgn_channel(num_symbols=10000):
    """
    Verify AWGN implementation
    Check: symbol error ratio ∝ 1/sqrt(SNR)
    """
    print("=" * 50)
    print("VERIFICATION OF AWGN IMPLEMENTATION")
    print("=" * 50)

    # Different SNR pairs to test
    test_pairs = [(10, 5), (15, 10), (20, 15), (15, 5)]

    results = []

    for SNR_B, SNR_E in test_pairs:
        channel = WiretapAWGNChannel(SNR_B, SNR_E)

        bob_errors = 0
        eve_errors = 0

        for _ in range(num_symbols):
            # Generate random 7-bit word
            x = [random.randint(0, 1) for _ in range(7)]
            y, z = channel.transmit(x)

            bob_errors += sum(1 for i in range(7) if x[i] != y[i])
            eve_errors += sum(1 for i in range(7) if x[i] != z[i])

        bob_error_rate = bob_errors / (num_symbols * 7)
        eve_error_rate = eve_errors / (num_symbols * 7)

        # Theoretical expectation: error rate ratio ~ sqrt(SNR_E/SNR_B)
        snr_ratio = 10 ** ((SNR_E - SNR_B) / 20)  # sqrt of linear ratio
        error_rate_ratio = eve_error_rate / bob_error_rate if bob_error_rate > 0 else 0

        results.append({
            'SNR_B': SNR_B,
            'SNR_E': SNR_E,
            'bob_error_rate': bob_error_rate,
            'eve_error_rate': eve_error_rate,
            'error_rate_ratio': error_rate_ratio,
            'theoretical_ratio': snr_ratio
        })

        print(f"\nSNR_B = {SNR_B} dB, SNR_E = {SNR_E} dB")
        print(f"  Bob error rate: {bob_error_rate:.6f}")
        print(f"  Eve error rate: {eve_error_rate:.6f}")
        print(f"  Error rate ratio (Eve/Bob): {error_rate_ratio:.4f}")
        print(f"  Theoretical sqrt(SNR_E/SNR_B): {snr_ratio:.4f}")

    return results

# Run verification
verification_results = verify_awgn_channel(num_symbols=5000)

VERIFICATION OF AWGN IMPLEMENTATION

SNR_B = 10 dB, SNR_E = 5 dB
  Bob error rate: 0.410000
  Eve error rate: 0.438714
  Error rate ratio (Eve/Bob): 1.0700
  Theoretical sqrt(SNR_E/SNR_B): 0.5623

SNR_B = 15 dB, SNR_E = 10 dB
  Bob error rate: 0.382400
  Eve error rate: 0.406914
  Error rate ratio (Eve/Bob): 1.0641
  Theoretical sqrt(SNR_E/SNR_B): 0.5623

SNR_B = 20 dB, SNR_E = 15 dB
  Bob error rate: 0.343629
  Eve error rate: 0.372514
  Error rate ratio (Eve/Bob): 1.0841
  Theoretical sqrt(SNR_E/SNR_B): 0.5623

SNR_B = 15 dB, SNR_E = 5 dB
  Bob error rate: 0.380029
  Eve error rate: 0.433371
  Error rate ratio (Eve/Bob): 1.1404
  Theoretical sqrt(SNR_E/SNR_B): 0.3162
